# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Load bacpipe and inspect the current configuration and settings, list all available API endpoints, supported models, and embedding dimensions.

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython


from IPython.display import display
import os 

# replace this with the path to the directory on your machine
os.chdir('../..')

import bacpipe

---
## 2. Using a Custom Model
Define your own model by subclassing `ModelBaseClass` and plug it directly into bacpipe's pipeline.

In [ ]:
import librosa as lb
from bacpipe.model_pipelines.model_utils import ModelBaseClass

class MyModel(ModelBaseClass):
    SAMPLE_RATE = 48000
    SEGMENT_LENGTH = 48000

    def __init__(self, **kwargs):
        super().__init__(sr=self.SAMPLE_RATE, segment_length=self.SEGMENT_LENGTH, **kwargs)

    def preprocess(self, audio):
        return audio * 10

    def __call__(self, audio):
        audio = audio.cpu().numpy()
        mel_spec = lb.feature.melspectrogram(y=audio, sr=self.SAMPLE_RATE)
        # return array needs to be 2D!
        mel_spec = mel_spec.reshape(
            [len(mel_spec), mel_spec.shape[-2] * mel_spec.shape[-1]]
            )
        return mel_spec

# Low-level approach
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')
embed_obj = bacpipe.Embedder('mymodel', loader_obj, device='cuda', CustomModel=MyModel)
mel_spectrograms = embed_obj.get_embeddings_from_model(loader_obj.files[0])

# High-level approach
loader_obj = bacpipe.generate_embeddings(
    model_name='mymodel',
    audio_dir='bacpipe/tests/test_data',
    CustomModel=MyModel
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings())

---
## 3. Extending an Existing Model
Subclass an existing bacpipe model to modify its behaviour — for example, squaring the input before passing it through BirdNET.

In [ ]:
from bacpipe.model_pipelines.feature_extractors.birdnet import Model

class MyBirdNETModel(Model):
    def __call__(self, input):
        input = input ** 2
        return self.embeds(input, training=False)

loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet2',
    audio_dir='bacpipe/tests/test_data',
    CustomModel=MyBirdNETModel
)

---
## 4. Full Pipeline — Single Model
Run the complete pipeline for a single model: embedding generation, classification, dimensionality reduction, and visualisation.

In [ ]:
# With a custom model
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet2',
    audio_dir='bacpipe/tests/test_data',
    CustomModel=MyBirdNETModel
)

---
## 5. Full Pipeline — Multiple Models
Run the pipeline across multiple models simultaneously, including custom ones, and compare their embeddings side by side.

In [ ]:
# Run pipeline for a mix of built-in and custom models
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'mymodel', 'birdnet2'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap',
    CustomModels=[None, MyModel, MyBirdNETModel],
    check_if_already_processed=False
)

---
## 6. End-to-End: `bacpipe.play`
The highest-level entry point — runs the full bacpipe pipeline including embeddings, classification, dimensionality reduction, evaluation, and dashboard visualisation in one call.

In [ ]:
bacpipe.play(
    models=['birdnet', 'mymodel', 'birdnet2'],
    audio_dir='bacpipe/tests/test_data',
    dim_reduction_model='umap',
    CustomModels=[None, MyModel, MyBirdNETModel]
)

---
## 7. Benchmarking Classifier Performance
Evaluate a model's classifier against ground truth annotations using precision, recall, and F1 score per species. Labels are matched using exact string lookup with a regex fallback for handling hyphens and spacing differences.

In [ ]:
# MyModel wont work because it doesn't have a classifier method, but birdnet2 does:
bacpipe.benchmark(
    'birdnet2', 
    'bacpipe/tests/test_data', 
    annotations_file='annotations.csv',
    CustomModel=MyBirdNETModel
)